# GSB 5544 — Topic 12.2: Extraction Walkthrough — Building the Week 12 Data Files from Their Sources
*Topic 12.1 starts from two cached CSV files. This read-along notebook shows where they came from and rebuilds them live, one step at a time. Four kinds of question appear in bold: **Explain, step by step** (read a cell aloud, line by line), **Write the code** (one or two lines, in the cell provided), **Fill in the blank** (`____` in code or prose) and **Discuss** (write a sentence or two). Needs the internet.*

## Part 0 — Where the Week 12 data came from

Topic 12.1 reads two files. Neither was typed in; each is the saved result of code that read a public source.

| File 12.1 reads | Source | What the source looks like | Permission |
|---|---|---|---|
| `hof_inductees.csv` (783 rows) | Wikipedia, [*List of Pro Football Hall of Fame inductees*](https://en.wikipedia.org/wiki/List_of_Pro_Football_Hall_of_Fame_inductees) | one web page, about 936 KB of HTML, 15 tables; the NFL table has one row per inductee **and team** | text licensed CC BY-SA 4.0; identify your script in the request |
| `nfl_rb_seasons_1999_2025.csv` (4,619 rows) | nflverse-data on GitHub, release [`stats_player`](https://github.com/nflverse/nflverse-data/releases/tag/stats_player) | one CSV per season, `stats_player_reg_<season>.csv`, about 2,000 players × 148 columns each | data released CC BY 4.0 by the nflverse project |

The outline's first choice, Pro Football Reference, was dropped: its terms restrict automated access and the use of
its content for machine-learning methods. **Checking permission is part of the Extract step**, before any code.

An extraction is finished when five things exist: the source (a URL you may use), the `____` that reads it, the
rows and columns kept, the saved `____`, and the access `____`. This notebook produces all five for both files.
It writes into a folder of its own, `week12_data/`, and at the end compares what it built with the repository's
cached copies.

**Discuss.** Topic 12.1 reads each file with one line of `pd.read_csv`. What is lost if a project *only* has that
line and the file, and not this notebook?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

In [ ]:
import io
import warnings
from datetime import date
from pathlib import Path

import pandas as pd
import requests

warnings.filterwarnings("ignore")

OUT = Path("week12_data")            # this notebook writes here; the repository's copies are not touched
OUT.mkdir(exist_ok=True)
CACHE = "https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/"   # the cached copies Topic 12.1 reads
print("writing to", OUT.resolve())

## Part 1 — Source 1: the Hall of Fame list (Wikipedia)

### 1.1 Fetch the page

A web page is text. `requests.get` asks the server for it and hands back a *response*; the HTML is in `.text`.

In [ ]:
HOF_URL = "https://en.wikipedia.org/wiki/List_of_Pro_Football_Hall_of_Fame_inductees"

response = requests.get(HOF_URL, headers={"User-Agent": "GSB5544-classroom-notebook/1.0"}, timeout=60)
print(response.status_code, "|", f"{len(response.text):,}", "characters of HTML")
response.text[:300]

**Explain, step by step.** Read the `requests.get(...)` line aloud: what are the three arguments, what does the
status code tell you, and what is in `response.text`?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### 1.2 From HTML to tables

`pd.read_html` finds every `<table>` in a page and turns each into a DataFrame. It accepts a file or a file-like
object, so the string is wrapped in `io.StringIO`, which makes a string behave like an open `____`.

In [ ]:
tables = pd.read_html(io.StringIO(response.text))

In [ ]:
# Write the code: print how many tables the page has, then the shape of each one



15 tables, most of them navigation boxes with a handful of rows. Two are large.

### 1.3 Find the right table by its column names, not its position

The inductee table could be `tables[1]` today and `tables[2]` after the next edit to the page. A test on the
**column names** fails loudly if the page changes; an index fails silently.

In [ ]:
want = {"Inductee", "Class", "Position", "Team(s)", "Years"}
hits = [i for i, t in enumerate(tables) if ____]
print("tables with those five columns:", hits)
for i in hits:
    print(i, tables[i].shape, list(tables[i].columns))

**Explain, step by step.** Read the `hits = [...]` line aloud. What does `enumerate` give, what does `issubset`
test, and why is the result a list of numbers?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

Two tables pass: table 1 (NFL inductees) and table 2 (156 rows of pre-NFL and
other-league careers, with a `League(s)` column). The project wants the NFL table, so the extra column is the
tie-breaker.

In [ ]:
nfl_table = [tables[i] for i in hits if ____][0]
hof = nfl_table[["Inductee", "Class", "Position", "Team(s)", "Years"]]
hof.shape

**Discuss.** Why select the five columns by name again, when the table already has exactly those five?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### 1.4 Look before you save

Saving is cheap; saving the wrong thing is expensive. Look at the head, the types and the repeats.

In [ ]:
print(hof.dtypes, "\n")
hof.head(8)

In [ ]:
# Write the code: count the rows, the distinct inductees, and show the five most repeated names



783 rows but 387 people: one row per inductee `____`, up to 9 rows for one
person. Names carry footnote marks (`Marcus Allen** [4]`), and `Position` mixes "Running back", "Halfback" and
long pre-modern labels. **None of that is fixed here.** The Extract step saves what the source said; cleaning
names, parsing years and collapsing to one row per person is `____` work and lives in Topic 12.1, Part 3,
where every change is visible and checkable.

### 1.5 Save it, with the date

In [ ]:
hof.to_csv(OUT / "hof_inductees.csv", ____)
(OUT / "nfl_hof_access_date.txt").write_text(date.today().isoformat() + "\n")
print("saved", OUT / "hof_inductees.csv", "| accessed", (OUT / "nfl_hof_access_date.txt").read_text().strip())

**Fill in the blank (prose).** Without `index=False`, re-reading the CSV would show an extra column called
`____` holding the old row numbers.

In [ ]:
# Write the code: re-read the saved file and check that it has the same shape as `hof`



### 1.6 Compare with the cached copy that Topic 12.1 reads

The repository's copy was saved on 2026-11-10. Wikipedia changes; the comparison says whether today's page
still agrees with it.

In [ ]:
cached_hof = pd.read_csv(CACHE + "hof_inductees.csv")
same = hof.reset_index(drop=True).astype(str).equals(cached_hof.astype(str))
print("live", hof.shape, "| cached", cached_hof.shape, "| identical:", same)

**Discuss.** Suppose this prints `False` next August. What happened, which copy should the project use, and what
does "refresh the data" have to include?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

## Part 2 — Source 2: season-by-season statistics (nflverse)

### 2.1 One file per season

The nflverse project publishes one CSV per season on a GitHub *release* page. The file names differ only in the
season, so one URL pattern with a `{season}` slot covers all of them.

In [ ]:
NFLVERSE_URL = "https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_reg_{season}.csv"
print(NFLVERSE_URL.format(season=____))

one = pd.read_csv(NFLVERSE_URL.format(season=2024), low_memory=False)
print(one.shape)
list(one.columns)[:30]

**Explain, step by step.** What does `.format(season=2024)` do to the string? Why read the file with
`pd.read_csv` straight from a URL? What is `low_memory=False` for?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

1,997 players and 148 columns for 2024 alone — passing, rushing, receiving, defence, kicking, punting,
fantasy points. The project needs a dozen of them.

In [ ]:
# Write the code: show which positions appear in the 2024 file and how many players each has



### 2.2 Which rows, which columns

Two decisions, made once and written down: keep the players whose listed position is a running back, and keep
the identifying columns plus the rushing and receiving counts.

In [ ]:
KEEP = ["player_id", "player_display_name", "position", "recent_team", "season", "games",
        "carries", "rushing_yards", "rushing_tds", "receptions", "targets", "receiving_yards", "receiving_tds"]

is_back = one["position"].isin(____)
one_rb = one.loc[is_back, KEEP]
print(one_rb.shape)
one_rb.sort_values("rushing_yards", ascending=False).head(5)

**Explain, step by step.** Read `one_rb = one.loc[is_back, KEEP]` aloud. What is `is_back`, and what do the two
things inside the square brackets each select?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

**Discuss.** Three choices to defend: (a) `player_id` is kept although `player_display_name` is there; (b)
`receptions`, `targets` and `receiving_yards` are kept although Topic 12.1 only plots rushing; (c) `"HB"` is in
the list although no 2024 player has it.

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### 2.3 All 27 seasons: the loop

The same three lines, once per season, collected in a list and stacked at the end. One line in the loop is a
**check**: if a season's file does not have the 13 columns, stop immediately and say which season and which
columns.

In [ ]:
FIRST_SEASON, LAST_SEASON = 1999, ____

frames = []
for season in range(FIRST_SEASON, ____):
    d = pd.read_csv(NFLVERSE_URL.format(season=season), low_memory=False)
    missing = [c for c in KEEP if c not in d.columns]
    assert not missing, (season, missing)          # stop loudly if a season's file has changed shape
    frames.append(d.loc[d["position"].isin(["RB", "HB", "FB"]), KEEP])
    print(season, f"{len(d):,} players ->", len(frames[-1]), "backs")

rb = pd.concat(frames, ____)
rb.shape

**Explain, step by step.** Read the loop aloud, one line at a time: the `range`, the read, the `missing` list,
the `assert`, the `append`, the `print`, and the `concat` after the loop.

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### 2.4 Check before saving

One row per player per season is the assumption every later step rests on. Say it as an `assert`, and look at the
counts.

In [ ]:
assert not rb.duplicated(["player_id", "season"]).any()      # one row per player per season
print(rb["player_id"].nunique(), "distinct players |", rb["season"].nunique(), "seasons")
rb["position"].value_counts()

In [ ]:
# Write the code: count the backs in each season and print the smallest and largest counts



**Discuss.** Fullbacks fall from 34 in 1999 to 6 in 2025. Does the extraction have a problem, and
what does the trend mean for Topic 12.1's 50-carry rule?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

### 2.5 Save, and compare with the cached copy

In [ ]:
rb.to_csv(OUT / "nfl_rb_seasons_1999_2025.csv", index=False)

cached_rb = pd.read_csv(CACHE + "nfl_rb_seasons_1999_2025.csv")
compare = pd.DataFrame({"live_rows":    rb.groupby("season").size(),
                        "cached_rows":  cached_rb.groupby("season").size(),
                        "live_yards":   rb.groupby("season")["rushing_yards"].sum(),
                        "cached_yards": cached_rb.groupby("season")["rushing_yards"].sum()})
differs = compare[(compare["live_rows"] != compare["cached_rows"]) | (compare["live_yards"] != compare["cached_yards"])]
print("live", rb.shape, "| cached", cached_rb.shape, "| seasons that differ:", len(differs))
differs

**Discuss.** The 2026 season is being played as you read this, yet the loop stops at 2025. Why not include the
current season, and what has to change when it ends?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

## Part 3 — What the Extract step leaves behind

In [ ]:
sorted(p.name for p in OUT.iterdir())

Three files: the two data sets and the access date. Together with this notebook they *are* the Extract step; Topic
12.1 can begin with `pd.read_csv` because they exist.

| What | Where | Why it matters |
|---|---|---|
| the source and its permission | Part 0 | someone else can judge whether the use was allowed |
| the code that read it | Parts 1 and 2 | the extraction can be repeated on purpose, not by accident |
| the rows and columns kept | 1.3 and 2.2 | the reader knows what was dropped on the way in |
| the saved files | `week12_data/` | every later cell reads the same bytes |
| the access `____` | `nfl_hof_access_date.txt` | "the page said so on 10 November" is a checkable claim |

**Discuss (closing).** Where does E end and T begin? The filter to running backs and the 13 columns happened
*here*; stripping `**` from the names and splitting `1993–1995` into two years happens in Topic 12.1. Why draw the
line there?

**Your answer:** *(think, then discuss with a neighbour — write a sentence or two here)*

---
## For your own final project — the Extract checklist

1. **Permission first.** Read the source's terms before writing code; write one sentence in the notebook saying
   what the terms allow. If the answer is no, find another source (the outline's first choice was replaced for
   exactly this reason).
2. **Identify yourself and fail fast.** A descriptive `User-Agent` and a `timeout` on every request.
3. **Find things by name, not by position.** Tables by their columns, files by a pattern, columns by a list.
4. **Look before saving.** `shape`, `dtypes`, `head`, repeats of the key.
5. **Check with `assert`.** One row per key; the columns you expect; stop loudly otherwise.
6. **Save what the source said** — no cleaning in the Extract step — with `index=False` and an access date.
7. **Compare with the cache** when you refresh, and re-read every sentence that quotes a number.